## Trained on Kaggle

In [ ]:
!pip install polars torchmetrics albumentations

In [ ]:
import os
from PIL import Image
import numpy as np
import json
import matplotlib.pyplot as plt
import polars as pl
import torch

In [ ]:
BASE = "/kaggle/input/datasets/orvile/bccd-blood-cell-count-and-detection-dataset"
TRAIN = "train"
TEST = "test"
VAL = "val"
IMG = "img"
ANN = "ann"
META = "meta.json"
EXT = (".json", ".jpeg")
IMG_SIZE = 512
SEED = 15002
DEVICE = 'cuda'
TRAIN_BATCH = 4
# BATCH = 48

In [ ]:
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# dataset

import os
import pickle
import polars as pl
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch
from PIL import Image

class BCCDDataset(Dataset):
    def __init__(self, df: pl.DataFrame, image_base: str, transforms = None):
        super().__init__()
        self.df = df
        self.image_base = image_base
        self.transforms = transforms

        self.image_ids = df["file_id"].unique().to_list()
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        
        _sub_df = self.df.filter(pl.col("file_id") == image_id)
        image_data = np.array(Image.open(os.path.join(self.image_base, _sub_df["image_file"][0])).convert("RGB"))
        
        boxes, labels = self._get_bbbox_labels(_sub_df)
        
        if self.transforms is not None:
            transformed = self.transforms(
                image=image_data,
                bboxes=boxes,
                labels=labels
            )
            image_data  = transformed['image']    # C x H x W tensor
            boxes  = transformed['bboxes']   # list(x1, y1, x2, y2)
            labels = transformed['labels']
        
        if len(boxes) == 0:
            boxes_tensor  = torch.zeros((0, 4), dtype=torch.float32)
            labels_tensor = torch.zeros((0,),   dtype=torch.int64)
            area_tensor   = torch.zeros((0,),   dtype=torch.float32)
        else:
            boxes_tensor  = torch.tensor(boxes,  dtype=torch.float32)
            labels_tensor = torch.tensor(labels, dtype=torch.int64)
            area_tensor   = (boxes_tensor[:, 2] - boxes_tensor[:, 0]) * \
                            (boxes_tensor[:, 3] - boxes_tensor[:, 1])
        
        target = {
            'boxes':    boxes_tensor,
            'labels':   labels_tensor,
            'area':     area_tensor,
            'image_id': torch.tensor([idx]),
            # iscrowd = 1 means the annotation covers a crowd of objects
            # and should be ignored during mAP computation.
            # BCCD has no crowd annotations, so this is always 0.
            'iscrowd':  torch.zeros((len(labels_tensor),), dtype=torch.int64)
        }

        return image_data, target
    
    def _get_bbbox_labels(self, df: pl.DataFrame):
        exp1 = (pl.col("xmin") < pl.col("xmax")).alias("xvalid")
        exp2 = (pl.col("ymin") < pl.col("ymax")).alias("yvalid")
        exp3 = (pl.col("xvalid") & pl.col("yvalid")).alias("valid")
        exp4 = pl.col("valid") == True
        
        boxes = []
        labels = []
        
        _sub_df = df.with_columns([
            exp1,
            exp2
        ]).with_columns(exp3).filter(exp4)
        
        for row in _sub_df.rows(named=True):
            boxes.append([row["xmin"], row["ymin"], row["xmax"], row["ymax"]])
            labels.append(row["label"])
        return boxes, labels


def collate_fn(batch):
    images  = [item[0] for item in batch] # this will be list of tensors
    targets = [item[1] for item in batch] # this will ne list of dicts: key -> tensor
    return images, targets

In [ ]:
# transforms

import albumentations as A
from albumentations.pytorch import ToTensorV2


def get_train_transforms(img_size=512):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        
        A.PadIfNeeded(min_height=img_size, min_width=img_size,
                      border_mode=0, fill=(123, 117, 104)),  # ImageNet mean color # padding is required as fix size is expected by model

        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2), # if we flip vertically it wont effect blood images

        # Color augmentations — only affect pixel values, not box coords
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, p=0.3),
        A.GaussianBlur(blur_limit=3, p=0.2),

        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()

    ], bbox_params=A.BboxParams(
        format='pascal_voc',      # input format is [x1, y1, x2, y2] in pixels
        label_fields=['labels'],
        clip=True, # clip boxes to image boundary after augmentation
        min_area=16, # drop boxes that become smaller than 16px² (noise)
        min_visibility=0.3 # drop boxes where <30% of area is still visible
    ))


def get_val_transforms(img_size=512):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size,
                      border_mode=0, fill=(123, 117, 104)),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ], bbox_params=A.BboxParams(
        format='pascal_voc',
        label_fields=['labels'],
        clip=True,
        min_area=16,
        min_visibility=0.3
    ))

In [ ]:
train_df = pl.read_csv("/kaggle/input/datasets/amallick0507/bccd-dfs/train_data.csv")
test_df = pl.read_csv("/kaggle/input/datasets/amallick0507/bccd-dfs/test_data.csv")
val_df = pl.read_csv("/kaggle/input/datasets/amallick0507/bccd-dfs/val_data.csv")
train_df.head(2)

In [ ]:
train_transform = get_train_transforms(IMG_SIZE)
val_transform = get_val_transforms(IMG_SIZE)

train_ds = BCCDDataset(train_df, image_base=os.path.join(BASE, TRAIN, IMG), transforms=train_transform)
test_ds = BCCDDataset(test_df, image_base=os.path.join(BASE, TEST, IMG), transforms=val_transform)
val_ds = BCCDDataset(val_df, image_base=os.path.join(BASE, VAL, IMG), transforms=val_transform)

train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH, shuffle=True, num_workers=0, collate_fn=collate_fn, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=1, num_workers=0, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=TRAIN_BATCH, num_workers=0, collate_fn=collate_fn, pin_memory=True)

In [ ]:
import torchvision
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.backbone_utils import BackboneWithFPN
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from torchvision.ops.feature_pyramid_network import LastLevelMaxPool
def build_mobilenet_fpn_backbone():
    backbone = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1)

    # MobileNetV3's feature layers are inside backbone.features
    # We extract intermediate layers by name for the FPN
    # These layer indices correspond to strides 8, 16, and 32
    return_layers = {
        '3':  '0',   # stride 8  — fine details, good for Platelets
        '7':  '1',   # stride 16 — medium features, good for RBCs
        '13': '2',   # stride 32 — coarse semantics, good for WBCs
    }

    # Channel sizes at each extracted layer in MobileNetV3-Large
    in_channels_list = [24, 80, 160]
    out_channels = 256  # FPN normalizes all levels to this channel count

    backbone_with_fpn = BackboneWithFPN(
        backbone=backbone.features,
        return_layers=return_layers,
        in_channels_list=in_channels_list,
        out_channels=out_channels,
        extra_blocks=LastLevelMaxPool()  # adds one more pooled level for very large objects
    )

    return backbone_with_fpn


def build_anchor_generator():    
    # Platelets are tiny: we need small anchors (16, 32 px)
    # RBCs are medium: 32, 64 px
    # WBCs are large: 64, 128, 256 px
    # Aspect ratios: all three cell types are roughly circular, so 0.5, 1.0, 2.0
    anchor_generator = AnchorGenerator(
        sizes=((16, 32), (32, 64), (64, 128), (128, 256)),
        aspect_ratios=((0.5, 1.0, 2.0),) * 4  # same ratios at every level
    )
    return anchor_generator


def build_model(num_classes):

    backbone         = build_mobilenet_fpn_backbone()
    anchor_generator = build_anchor_generator()

    # RoI Align output size — each proposed region gets cropped to 7x7
    roi_pooler = torchvision.ops.MultiScaleRoIAlign(
        featmap_names=['0', '1', '2', '3'],  # which FPN levels to pool from
        output_size=7,
        sampling_ratio=2
    )

    model = FasterRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=roi_pooler,

        # RPN hyperparameters — these control how proposals are filtered
        rpn_pre_nms_top_n_train=2000,   # keep top 2000 proposals before NMS during training
        rpn_pre_nms_top_n_test=1000,
        rpn_post_nms_top_n_train=1000,  # keep top 1000 after NMS during training
        rpn_post_nms_top_n_test=300,
        rpn_nms_thresh=0.7,             # IoU threshold for RPN NMS
        rpn_fg_iou_thresh=0.7,          # anchor is "foreground" if IoU with GT > 0.7
        rpn_bg_iou_thresh=0.3,          # anchor is "background" if IoU with GT < 0.3
        rpn_batch_size_per_image=256,
        rpn_positive_fraction=0.5,

        # Detection head hyperparameters
        box_score_thresh=0.05,     # low threshold during training to see all predictions
        box_nms_thresh=0.5,        # IoU threshold for final NMS
        box_detections_per_img=100 # max 100 detections per image
    )

    return model


def get_model_objectdetection_mobilenet(num_classes=4, device='cpu'):
    model = build_model(num_classes)
    model = model.to(device)

    # Count parameters as a sanity check
    total  = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total:,}")
    print(f"Trainable parameters: {trainable:,}")

    return model

In [ ]:
from tqdm import tqdm


def train_one_epoch(model, loader, optimizer, device, scaler):
    model.train()

    total_losses = []
    cls_losses = []
    box_reg_losses = []
    rpn_obj_losses = []
    rpn_box_reg_losses = []

    for images, targets in tqdm(loader, desc='Training'):
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()

        if scaler is not None:
            # Mixed precision: compute forward pass in float16 for speed
            with torch.amp.autocast(device):
                loss_dict = model(images, targets)
                losses    = sum(loss for loss in loss_dict.values())
            scaler.scale(losses).backward()
            # Unscale before clipping so clip threshold is in real gradient space
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            losses.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        total_losses.append(losses.item())
        cls_losses.append(loss_dict['loss_classifier'].item())
        box_reg_losses.append(loss_dict['loss_box_reg'].item())
        rpn_obj_losses.append(loss_dict['loss_objectness'].item())
        rpn_box_reg_losses.append(loss_dict['loss_rpn_box_reg'].item())

    n = len(loader)
    return {
        'total': sum(total_losses)/ n,
        'cls': sum(cls_losses)/ n,
        'box_reg': sum(box_reg_losses)/ n,
        'rpn_obj': sum(rpn_obj_losses)/ n,
        'rpn_box_reg': sum(rpn_box_reg_losses)/ n,
    }


@torch.no_grad()
def evaluate_loss(model, loader, device):
    model.train()  # needed to get loss_dict — yes this looks wrong, it's correct
    total_losses = []

    for images, targets in tqdm(loader, desc='Val Loss'):
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        with torch.amp.autocast(device):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

        total_losses.append(losses.item())

    return sum(total_losses) / len(loader)

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm import tqdm


# @torch.no_grad()
# def evaluate_map(model, loader, device, score_thresh=0.5):
#     model.eval()

#     metric = MeanAveragePrecision(
#         iou_type='bbox',
#         class_metrics=True  # gives per-class AP breakdown
#     )

#     for images, targets in tqdm(loader, desc='Computing mAP'):
#         images  = [img.to(device) for img in images]
#         targets_device = [{k: v.to(device) for k, v in t.items()} for t in targets]

#         predictions = model(images)

#         # torchmetrics expects predictions and targets as lists of dicts
#         # Predictions dict needs: boxes, scores, labels
#         # Targets dict needs:    boxes, labels
#         preds_fmt = []
#         for pred in predictions:
#             preds_fmt.append({
#                 'boxes':  pred['boxes'].cpu(),
#                 'scores': pred['scores'].cpu(),
#                 'labels': pred['labels'].cpu()
#             })

#         tgts_fmt = []
#         for tgt in targets:
#             tgts_fmt.append({
#                 'boxes':  tgt['boxes'].cpu(),
#                 'labels': tgt['labels'].cpu()
#             })

#         metric.update(preds_fmt, tgts_fmt)

#     results = metric.compute()

#     print(f"\nmAP@50:95 = {results['map']:.4f}")
#     print(f"mAP@50 = {results['map_50']:.4f}")
#     print(f"mAP@75 = {results['map_75']:.4f}")

#     if 'map_per_class' in results:
#         print(results['map_per_class'])
#         class_names = ['Platelets', 'RBC', 'WBC', 'background']
#         print("\nPer-class AP@50:95:")
#         for i, ap in enumerate(results['map_per_class']):
#             # class indices in torchmetrics start at 1 for your classes
#             print(f"{class_names[i+1]:12s}: {ap:.4f}")

#     return results

@torch.no_grad()
def evaluate_map(model, loader, device, score_thresh=0.5):
    model.eval()
    metric = MeanAveragePrecision(
        iou_type='bbox',
        class_metrics=True
    )

    for images, targets in tqdm(loader, desc='Computing mAP'):
        images = [img.to(device) for img in images]

        predictions = model(images)

        preds_fmt = [{
            'boxes':  pred['boxes'].cpu(),
            'scores': pred['scores'].cpu(),
            'labels': pred['labels'].cpu()
        } for pred in predictions]

        tgts_fmt = [{
            'boxes':  tgt['boxes'].cpu(),
            'labels': tgt['labels'].cpu()
        } for tgt in targets]                   # use targets directly, not targets_device

        metric.update(preds_fmt, tgts_fmt)

    results = metric.compute()

    print(f"\nmAP@50:95 = {results['map']:.4f}")
    print(f"mAP@50    = {results['map_50']:.4f}")
    print(f"mAP@75    = {results['map_75']:.4f}")

    if 'map_per_class' in results:
        # map_per_class returns one value per class in the order
        # torchmetrics encounters them — matches your label indices 0,1,2
        CLASS_NAMES = ['Platelets', 'RBC', 'WBC']   # index 0, 1, 2
        print("\nPer-class AP@50:95:")
        for i, ap in enumerate(results['map_per_class']):
            print(f"{CLASS_NAMES[i]:12s}: {ap:.4f}")

    return results

In [ ]:
model = get_model_objectdetection_mobilenet(num_classes=4, device=DEVICE)
history = []
best_map = 0.0

In [ ]:
# phase 1

# freeze backbone
for param in model.backbone.parameters():
    param.requires_grad = False

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3, weight_decay=1e-4
)
scaler = torch.amp.GradScaler(DEVICE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
print(f"{"="*20}")
print("Training start: Phase 1")

for epoch in range(10):
    train_metrics = train_one_epoch(model, train_loader, optimizer, DEVICE, scaler)
    val_loss = evaluate_loss(model, val_loader, DEVICE)
    val_map = evaluate_map(model, val_loader, DEVICE)
    scheduler.step()

    history.append({
        'epoch': epoch + 1,
        'phase': 1,
        'train_loss': train_metrics['total'],
        'val_loss': val_loss,
        'val_map': val_map['map'],
        'val_map_50': val_map['map_50'],
    })

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {train_metrics['total']:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"mAP@50: {val_map['map_50']:.4f} | "
          f"mAP@50:95: {val_map['map']:.4f}")

    if val_map['map_50'] > best_map:
        best_map = val_map['map_50']
        torch.save(model.state_dict(), 'best_phase1.pth')
        print(f"New best mAP@50: {best_map:.4f}")



In [ ]:
# phase 2
for param in model.backbone.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-4}, 
    {'params': model.rpn.parameters(), 'lr': 5e-4},
    {'params': model.roi_heads.parameters(), 'lr': 5e-4},
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

print(f"{"="*20}")
print("Training start: Phase 1")
for epoch in range(20):
    train_metrics = train_one_epoch(model, train_loader, optimizer, DEVICE, scaler)
    val_loss = evaluate_loss(model, val_loader, DEVICE)
    val_map = evaluate_map(model, val_loader, DEVICE)
    scheduler.step()

    history.append({
        'epoch': epoch + 11,
        'phase': 2,
        'train_loss': train_metrics['total'],
        'val_loss': val_loss,
        'val_map': val_map['map'],
        'val_map_50': val_map['map_50'],
    })

    print(f"Epoch {epoch+11:02d} | "
          f"Train Loss: {train_metrics['total']:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"mAP@50: {val_map['map_50']:.4f} | "
          f"mAP@50:95: {val_map['map']:.4f}")

    if val_map['map_50'] > best_map:
        best_map = val_map['map_50']
        torch.save(model.state_dict(), 'best_final.pth')
        print(f"New best mAP@50: {best_map:.4f}")

In [ ]:
import matplotlib.patches as patches
CLASS_COLORS = {
    1: ('#FF4444', 'RBC'),
    2: ('#4444FF', 'WBC'),
    3: ('#FFDD00', 'Platelets')
}


def denormalize(tensor):
    """Reverse ImageNet normalization for display."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


@torch.no_grad()
def visualize_predictions(model, dataset, device, n=5, score_thresh=0.5):
    model.eval()
    indices = torch.randperm(len(dataset))[:n].tolist()

    fig, axes = plt.subplots(1, n, figsize=(10 * n, 10))
    if n == 1:
        axes = [axes]

    for ax, idx in zip(axes, indices):
        image, target = dataset[idx]

        pred = model([image.to(device)])[0]

        img_np = denormalize(image.cpu())
        ax.imshow(img_np)

        # Draw ground truth boxes — solid lines
        for box, label in zip(target['boxes'], target['labels']):
            x1, y1, x2, y2 = box.tolist()
            color, name    = CLASS_COLORS.get(label.item(), ('#FFFFFF', 'unknown'))
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor=color, facecolor='none', linestyle='-'
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, f'GT:{name}', color=color, fontsize=7, weight='bold')

        # Draw predicted boxes — dashed lines, only above score threshold
        for box, label, score in zip(pred['boxes'], pred['labels'], pred['scores']):
            if score < score_thresh:
                continue
            x1, y1, x2, y2 = box.tolist()
            color, name    = CLASS_COLORS.get(label.item(), ('#FFFFFF', 'unknown'))
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor=color, facecolor='none', linestyle='--'
            )
            ax.add_patch(rect)
            ax.text(x1, y2 + 10, f'{name}:{score:.2f}',
                    color=color, fontsize=7, weight='bold')

        n_gt   = len(target['boxes'])
        n_pred = (pred['scores'] >= score_thresh).sum().item()
        ax.set_title(f'GT: {n_gt} boxes | Pred: {n_pred} boxes', fontsize=9)
        ax.axis('off')

    plt.suptitle('Solid = Ground Truth  |  Dashed = Predictions', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
len(test_loader)

In [ ]:
visualize_predictions(model, test_ds, DEVICE, n = 2)

In [ ]:
score_thresh = 0.5
for images, targets in tqdm(test_loader, desc='Predicting on test'):
    images = [img.to(DEVICE) for img in images]
    targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

    pred = model([images[0].to(DEVICE)])[0]

    img_np = denormalize(images[0].cpu())

    fig, axes = plt.subplots(1, 1, figsize=(10 , 10))
    axes.imshow(img_np)
    target = targets[0]
    id_idx = int(target['image_id'][0].cpu())
    filename = test_ds.image_ids[id_idx]
    # Draw ground truth boxes — solid lines
    for box, label in zip(target['boxes'], target['labels']):
        x1, y1, x2, y2 = box.tolist()
        color, name    = CLASS_COLORS.get(label.item(), ('#FFFFFF', 'unknown'))
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none', linestyle='-'
        )
        axes.add_patch(rect)
        axes.text(x1, y1 - 4, f'GT:{name}', color=color, fontsize=7, weight='bold')

    # Draw predicted boxes — dashed lines, only above score threshold
    for box, label, score in zip(pred['boxes'], pred['labels'], pred['scores']):
        if score < score_thresh:
            continue
        x1, y1, x2, y2 = box.tolist()
        color, name    = CLASS_COLORS.get(label.item(), ('#FFFFFF', 'unknown'))
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none', linestyle='--'
        )
        axes.add_patch(rect)
        axes.text(x1, y2 + 10, f'{name}:{score:.2f}',
                color=color, fontsize=7, weight='bold')

    n_gt   = len(target['boxes'])
    n_pred = (pred['scores'] >= score_thresh).sum().item()
    axes.set_title(f'GT: {n_gt} boxes | Pred: {n_pred} boxes', fontsize=9)
    axes.axis('off')
    plt.suptitle('Solid = Ground Truth  |  Dashed = Predictions', fontsize=12)
    plt.tight_layout()
    # plt.show()
    plt.savefig(f'{filename}_predictions.png', dpi=150, bbox_inches='tight')